# Game, Set, Match! — Tennis Stats Linear Regression
## Extended Solution Notebook

**Codecademy Project (extended)**  
Predict ATP player outcomes (Winnings, Wins, Ranking) from service & return statistics using linear regression.

**Data:** `tennis_stats.csv` — 1 721 player-year observations (2009–2017), top ~1500 ranked players.

**Notebook goals**
1. Load & investigate the data  
2. Exploratory analysis (scatter plots + correlation)  
3. Single-feature linear regression models  
4. Two-feature models for yearly earnings  
5. Multi-feature models  
6. Alternate implementations  
7. More practice exercises  
8. Simulation / sensitivity section  

![Flowchart](tennis_stats_flowchart.png)


## 1. Load and Investigate the Data

Load `tennis_stats.csv` into a pandas DataFrame. Examine shape, columns, dtypes, missing values and basic statistics.  
Pay special attention to the outcome columns: **Wins**, **Losses**, **Winnings**, **Ranking**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# optional nicer plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

df = pd.read_csv('tennis_stats.csv')
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nFirst 5 rows:')
display(df.head())
print('\nInfo:')
print(df.info())
print('\nDescribe (numeric):')
display(df.describe().round(3))
print('\nMissing values total:', df.isnull().sum().sum())


## 2. Exploratory Analysis

Plot several features against the three main outcomes.  
Look for linear relationships. Compute a correlation matrix focused on outcomes.


In [ ]:
# Quick correlation view with outcomes
outcomes = ['Wins', 'Winnings', 'Ranking']
features_of_interest = [
    'Aces', 'BreakPointsOpportunities', 'ServiceGamesPlayed',
    'ReturnGamesPlayed', 'FirstServePointsWon', 'SecondServePointsWon',
    'ReturnPointsWon', 'TotalPointsWon', 'DoubleFaults'
]

corr = df[features_of_interest + outcomes].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr.loc[features_of_interest, outcomes], annot=True, cmap='RdYlGn', center=0, fmt='.2f')
plt.title('Feature ↔ Outcome Correlations')
plt.tight_layout()
plt.savefig('tennis_corr_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nStrongest correlations with Winnings:')
print(df.corr(numeric_only=True)['Winnings'].sort_values(ascending=False).head(12))


In [ ]:
# Scatter plots: a few key features vs Winnings
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, feat in enumerate(['BreakPointsOpportunities', 'ServiceGamesPlayed', 'Aces',
                          'FirstServePointsWon', 'ReturnPointsWon', 'TotalPointsWon']):
    axes[i].scatter(df[feat], df['Winnings'], alpha=0.35, s=12)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Winnings')
    axes[i].set_title(f'{feat} vs Winnings')
plt.tight_layout()
plt.savefig('tennis_scatters_winnings.png', dpi=120, bbox_inches='tight')
plt.show()


## 3. Single-Feature Linear Regression

Choose **one** strong feature (e.g. `BreakPointsOpportunities`) and predict `Winnings`.  
Split into train/test (80/20), fit `LinearRegression`, evaluate on the held-out test set, and plot predictions vs actuals.


In [ ]:
# --- Single feature model ---
feature = 'BreakPointsOpportunities'
target = 'Winnings'

X = df[[feature]]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(f'Feature: {feature}')
print(f'Coefficient: {model.coef_[0]:.2f}')
print(f'Intercept: {model.intercept_:.2f}')
print(f'Test R²: {r2:.4f}')
print(f'Test MSE: {mse:,.0f}')

# Plot
plt.figure(figsize=(7, 5))
plt.scatter(X_test, y_test, alpha=0.4, label='Actual', s=20)
plt.plot(X_test, y_pred, color='red', linewidth=2, label='Predicted')
plt.xlabel(feature)
plt.ylabel(target)
plt.title(f'Single-Feature LR: {feature} → {target}\nTest R² = {r2:.3f}')
plt.legend()
plt.tight_layout()
plt.savefig('tennis_single_feature_pred.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. Compare Several Single-Feature Models

Train a simple linear model for each of several candidate features. Rank them by test R².  
Which single feature is the strongest predictor of yearly Winnings?


In [ ]:
candidates = [
    'BreakPointsOpportunities', 'ServiceGamesPlayed', 'ReturnGamesPlayed',
    'Aces', 'DoubleFaults', 'FirstServePointsWon', 'SecondServePointsWon',
    'ReturnPointsWon', 'TotalPointsWon', 'ServiceGamesWon'
]

results = []
for feat in candidates:
    X = df[[feat]]
    y = df['Winnings']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    m = LinearRegression().fit(X_tr, y_tr)
    r2 = r2_score(y_te, m.predict(X_te))
    results.append({'feature': feat, 'test_R2': r2, 'coef': m.coef_[0]})

single_df = pd.DataFrame(results).sort_values('test_R2', ascending=False)
print(single_df.to_string(index=False))
print('\nBest single feature:', single_df.iloc[0]['feature'])


## 5. Two-Feature Models for Yearly Earnings

Try several pairs of features. Which combination yields the highest test R² for predicting Winnings?


In [ ]:
pairs = [
    ['BreakPointsOpportunities', 'ServiceGamesPlayed'],
    ['BreakPointsOpportunities', 'Aces'],
    ['ServiceGamesPlayed', 'Aces'],
    ['BreakPointsOpportunities', 'ReturnGamesPlayed'],
    ['FirstServePointsWon', 'SecondServePointsWon'],
    ['Aces', 'DoubleFaults'],
    ['TotalPointsWon', 'BreakPointsOpportunities'],
]

two_results = []
for pair in pairs:
    X = df[pair]
    y = df['Winnings']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    m = LinearRegression().fit(X_tr, y_tr)
    r2 = r2_score(y_te, m.predict(X_te))
    two_results.append({'features': ' + '.join(pair), 'test_R2': r2})

two_df = pd.DataFrame(two_results).sort_values('test_R2', ascending=False)
print(two_df.to_string(index=False))
print('\nBest two-feature set:', two_df.iloc[0]['features'])


## 6. Multi-Feature Models

Build models that use 3+ features. Experiment with different subsets (volume stats, percentage stats, mixed).  
Report the best test R² you obtain for predicting Winnings.


In [ ]:
# Volume-heavy set (strongest)
volume_feats = ['BreakPointsOpportunities', 'ServiceGamesPlayed', 'ReturnGamesPlayed',
                'Aces', 'DoubleFaults', 'BreakPointsFaced']

# Percentage / quality set
pct_feats = ['FirstServePointsWon', 'SecondServePointsWon', 'ReturnPointsWon',
             'ServiceGamesWon', 'BreakPointsConverted', 'TotalPointsWon']

# Mixed
mixed_feats = ['BreakPointsOpportunities', 'ServiceGamesPlayed', 'Aces',
               'FirstServePointsWon', 'ReturnPointsWon', 'DoubleFaults']

multi_results = []
for name, feats in [('volume', volume_feats), ('percentage', pct_feats), ('mixed', mixed_feats)]:
    X = df[feats]
    y = df['Winnings']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    m = LinearRegression().fit(X_tr, y_tr)
    r2 = r2_score(y_te, m.predict(X_te))
    multi_results.append({'set': name, 'n_features': len(feats), 'test_R2': r2})
    print(f'{name:12s} R² = {r2:.4f}  features = {feats}')

multi_df = pd.DataFrame(multi_results).sort_values('test_R2', ascending=False)
print('\nBest multi-feature set:', multi_df.iloc[0]['set'])


In [ ]:
# Full multi-feature model with coefficient inspection
best_feats = volume_feats  # or the winner above
X = df[best_feats]
y = df['Winnings']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model_multi = LinearRegression().fit(X_tr, y_tr)
y_pred_multi = model_multi.predict(X_te)

print('Multi-feature coefficients:')
for f, c in zip(best_feats, model_multi.coef_):
    print(f'  {f:30s} {c:12.2f}')
print(f'Intercept: {model_multi.intercept_:.2f}')
print(f'Test R²: {r2_score(y_te, y_pred_multi):.4f}')

# Pred vs actual
plt.figure(figsize=(6, 6))
plt.scatter(y_te, y_pred_multi, alpha=0.4, s=15)
plt.plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--', lw=2)
plt.xlabel('Actual Winnings')
plt.ylabel('Predicted Winnings')
plt.title(f'Multi-feature model\nTest R² = {r2_score(y_te, y_pred_multi):.3f}')
plt.tight_layout()
plt.savefig('tennis_multi_pred_vs_actual.png', dpi=120, bbox_inches='tight')
plt.show()


## 7. Alternate Code / Implementations

### 7.1 Pipeline + StandardScaler
### 7.2 Predict Wins instead of Winnings
### 7.3 Manual OLS via normal equations (numpy)


In [ ]:
# 7.1 Pipeline with scaling (useful when features have very different scales)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])
X = df[volume_feats]
y = df['Winnings']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_tr, y_tr)
print('Pipeline (scaled) Test R²:', r2_score(y_te, pipe.predict(X_te)))
print('Note: coefficients are now on standardized scale')
print(list(zip(volume_feats, pipe.named_steps['lr'].coef_.round(1))))


In [ ]:
# 7.2 Same features predicting Wins
X = df[volume_feats]
y = df['Wins']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
m_wins = LinearRegression().fit(X_tr, y_tr)
print('Predicting Wins — Test R²:', r2_score(y_te, m_wins.predict(X_te)))


In [ ]:
# 7.3 Pure-numpy OLS (normal equations) — educational alternate
def ols_numpy(X, y):
    """Add intercept column and solve (X'X)^{-1} X'y"""
    X_ = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.lstsq(X_, y, rcond=None)[0]
    return beta  # intercept + coefficients

X_np = df[['BreakPointsOpportunities', 'ServiceGamesPlayed']].values
y_np = df['Winnings'].values
beta = ols_numpy(X_np, y_np)
print('Numpy OLS coefficients (intercept, BP_Opp, ServGames):', beta.round(2))


## 8. More Practice

1. Build a model that predicts **Ranking** (remember lower is better). Which features help most?  
2. Try a **polynomial** feature expansion of a single strong predictor.  
3. Use **cross-validation** (`cross_val_score`) to get a more robust estimate of R².  
4. Investigate whether **Year** adds predictive power (trend over time).


In [ ]:
# Practice 1 – Ranking
from sklearn.model_selection import cross_val_score

X = df[volume_feats]
y = df['Ranking']
scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')
print('5-fold CV R² for Ranking:', scores.round(3), 'mean =', scores.mean().round(3))

# Practice 2 – Polynomial
from sklearn.preprocessing import PolynomialFeatures
poly = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr', LinearRegression())
])
X1 = df[['BreakPointsOpportunities']]
y = df['Winnings']
X_tr, X_te, y_tr, y_te = train_test_split(X1, y, test_size=0.2, random_state=42)
poly.fit(X_tr, y_tr)
print('Polynomial (deg=2) single-feature Test R²:', r2_score(y_te, poly.predict(X_te)))


## 9. Simulation / Sensitivity Analysis

Modify key parameters and observe how model performance changes.  
This section is deliberately interactive — change the values in the parameter cell and re-run.


In [ ]:
# === SIMULATION PARAMETERS (edit these) ===
SIM_FEATURES = ['BreakPointsOpportunities', 'ServiceGamesPlayed', 'Aces', 'DoubleFaults']
SIM_TARGET   = 'Winnings'
TEST_SIZE    = 0.25          # try 0.1, 0.3, 0.5
RANDOM_SEED  = 7
N_BOOTSTRAP  = 80            # number of Monte-Carlo resamples
NOISE_SCALE  = 0.0           # add Gaussian noise to target (0 = none, try 0.1 * y.std())
SUBSAMPLE_FRAC = 1.0         # use only this fraction of data (try 0.3, 0.6)

# =========================================
rng = np.random.RandomState(RANDOM_SEED)

X_full = df[SIM_FEATURES].values
y_full = df[SIM_TARGET].values.astype(float)

# optional noise
if NOISE_SCALE > 0:
    y_full = y_full + rng.normal(0, NOISE_SCALE, size=len(y_full))

r2_list = []
for i in range(N_BOOTSTRAP):
    # subsample rows
    n = int(len(df) * SUBSAMPLE_FRAC)
    idx = rng.choice(len(df), size=n, replace=True)
    Xb, yb = X_full[idx], y_full[idx]
    
    X_tr, X_te, y_tr, y_te = train_test_split(Xb, yb, test_size=TEST_SIZE, random_state=rng.randint(0, 10_000))
    m = LinearRegression().fit(X_tr, y_tr)
    r2_list.append(r2_score(y_te, m.predict(X_te)))

r2_arr = np.array(r2_list)
print(f'Simulation settings: features={SIM_FEATURES}')
print(f'test_size={TEST_SIZE}, subsample={SUBSAMPLE_FRAC}, noise_scale={NOISE_SCALE}')
print(f'Monte-Carlo Test R²  mean = {r2_arr.mean():.4f}  std = {r2_arr.std():.4f}')
print(f'5%–95% interval     = [{np.percentile(r2_arr,5):.4f}, {np.percentile(r2_arr,95):.4f}]')

plt.figure(figsize=(7, 4))
plt.hist(r2_arr, bins=20, edgecolor='k', alpha=0.75, color='steelblue')
plt.axvline(r2_arr.mean(), color='red', lw=2, label=f'mean={r2_arr.mean():.3f}')
plt.xlabel('Test R²')
plt.ylabel('Count')
plt.title('Monte-Carlo distribution of Test R²\n(under chosen simulation parameters)')
plt.legend()
plt.tight_layout()
plt.savefig('tennis_simulation_r2_hist.png', dpi=120, bbox_inches='tight')
plt.show()


### Quick “what-if” experiments you can try
- Increase `NOISE_SCALE` to 30 000 → watch R² drop  
- Set `SUBSAMPLE_FRAC = 0.2` → higher variance in the R² histogram  
- Swap `SIM_TARGET` to `'Wins'` or `'Ranking'`  
- Replace `SIM_FEATURES` with percentage-only features and compare mean R²


## Key Findings (Solution)

- **Volume statistics** (games played, break-point opportunities, aces, double faults) are far stronger predictors of Winnings and Wins than pure percentage statistics.  
- A single feature such as `BreakPointsOpportunities` or `ServiceGamesPlayed` already yields Test R² ≈ 0.80–0.83.  
- Adding a second complementary volume feature pushes R² a little higher; a carefully chosen multi-feature set reaches ~0.85–0.87.  
- Ranking is harder to predict (lower R²) because ranking is an ordinal / relative measure and also depends on the rest of the field.  
- The simulation shows that R² estimates are reasonably stable once the sample exceeds a few hundred rows; heavy noise or tiny subsamples increase variance dramatically.

These results highlight that, in professional tennis, **playing a lot of high-level matches** (and therefore generating many break-point opportunities, aces, etc.) is tightly linked to higher earnings.
